# Imports

In [ ]:
import torch
import sys
import os
from torch_scatter import scatter
from torch import Tensor
sys.path.append('../..')
import pandas as pd
import plotly.express as px
import plotly.graph_objs as go
from definitions import EXPERIMENTS_DIR, ROOT_DIR
from src.commands.train import Trainer, Config
from torch_geometric.data import HeteroData
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    hamming_loss,
)
from PIL import ImageColor

In [2]:
px.defaults.template = 'seaborn'

In [3]:
dir = os.path.join(ROOT_DIR, 'assets', 'articles_classification_results')

# Load the results

In [5]:
# TASK = 'hierarchical_classification'
EXPERIMENT = 'health_care'
CORPUS = 'health-care'
EXPERIMENT_DIR = os.path.join(EXPERIMENTS_DIR, EXPERIMENT)
RESULTS_DIR = os.path.join(EXPERIMENT_DIR, 'results')
HISTORIES_DIR = os.path.join(EXPERIMENT_DIR, 'histories')
CONFIG_PATH = os.path.join(EXPERIMENT_DIR, 'config.json')
SPLIT_PATH = os.path.join(EXPERIMENT_DIR, 'split_nantes_uni_2020_2026_e5_4.pt')

In [6]:
history = pd.concat([
    pd.read_csv(os.path.join(HISTORIES_DIR, filename))
    for filename in os.listdir(HISTORIES_DIR)
])

histohealth_carery = history \
    .groupby(['epoch','phase','config','trial']) \
    .mean() \
    .reset_index()
history.head()

,epoch,phase,subdomain_guidance_loss,lab_guidance_loss,loss,subdomain_guidance_acc,subdomain_guidance_f1,lab_guidance_acc,lab_guidance_f1,config,trial
0,0,train,0.721110,0.718654,1.439764,0.428846,0.070454,0.418155,0.028614,Conv_Mean_null_False,3
1,0,train,0.606332,0.539432,1.145764,0.769231,0.046899,0.928571,0.044561,Conv_Mean_null_False,3
2,0,train,0.488302,0.398632,0.886934,0.906731,0.014377,0.971726,0.000000,Conv_Mean_null_False,3
3,0,train,0.391016,0.285645,0.676661,0.930769,0.018648,0.971726,0.000000,Conv_Mean_null_False,3
4,0,train,0.260188,0.206451,0.466639,0.961538,0.007692,0.968750,0.000000,Conv_Mean_null_False,3


In [7]:
history_mean = history \
    .drop(columns = 'trial') \
    .groupby(['epoch','phase','config']).mean() \
    .add_suffix('_mean')

history_std = history \
    .drop(columns = 'trial') \
    .groupby(['epoch','phase','config']).std() \
    .add_suffix('_std')

In [8]:
history = history_mean.join(other = history_std).reset_index()
history.head()

,epoch,phase,config,subdomain_guidance_loss_mean,lab_guidance_loss_mean,loss_mean,subdomain_guidance_acc_mean,subdomain_guidance_f1_mean,lab_guidance_acc_mean,lab_guidance_f1_mean,subdomain_guidance_loss_std,lab_guidance_loss_std,loss_std,subdomain_guidance_acc_std,subdomain_guidance_f1_std,lab_guidance_acc_std,lab_guidance_f1_std
0,0,train,Conv_Mean_null_False,0.138662,0.092925,0.231587,0.952718,0.073386,0.970599,0.048357,0.060782,0.052145,0.108580,0.031434,0.042357,0.028469,0.034023
1,0,val,Conv_Mean_null_False,0.116647,0.075398,0.192045,0.957262,0.100986,0.974161,0.066218,0.062012,0.016236,0.062707,0.026468,0.030650,0.006026,0.041873
2,1,train,Conv_Mean_null_False,0.108351,0.069862,0.178214,0.959584,0.137413,0.975685,0.095297,0.025882,0.013385,0.030913,0.010927,0.036599,0.005172,0.036234
3,1,val,Conv_Mean_null_False,0.109837,0.071068,0.180906,0.959123,0.132402,0.975461,0.080634,0.052984,0.015258,0.054238,0.023452,0.032718,0.005579,0.049525
4,2,train,Conv_Mean_null_False,0.100070,0.063168,0.163238,0.962121,0.161863,0.977796,0.114930,0.024217,0.013117,0.030099,0.010554,0.039595,0.005158,0.039333


# Load Data

In [9]:
config = Config(experiment=EXPERIMENT)
trainer = Trainer(experiment=config.experiment, config=config.training_config)
configs = trainer.get_configs()

In [10]:
data = trainer.load_data()
# data = trainer.filter_data(data)
# data = trainer.keep_largest_cc(data) if trainer.config.keep_largest_cc else data
data.split = trainer.create_or_load_split(data)
data['Article'].split = data.split

In [11]:
metadata = data.raw_metadata

# Learning Curves

In [12]:
def line_plot(
    df : pd.DataFrame,
    metric : str
) -> None:
    
    fig = go.Figure()

    colors = px.colors.qualitative.Plotly
    exprs = df['config'].unique()

    for i, expr in enumerate(exprs):

        history_val = df.query(f"phase == 'val' and config == '{expr}'")
        color = colors[i % len(colors)]

        fig.add_trace(
            go.Scatter(
                name = expr,
                x=history_val['epoch'],
                y=history_val[f'{metric}_mean'],
                mode='lines',
                line=dict(color=color),
            )
        )

        fig.add_trace(
            go.Scatter(
                x=history_val['epoch'],
                y=history_val[f'{metric}_mean']+history_val[f'{metric}_std'],
                mode='lines',
                marker=dict(color="#444"),
                line=dict(width=0),
                showlegend=False
            )
        )

        fig.add_trace(
            go.Scatter(
                x=history_val['epoch'],
                y=history_val[f'{metric}_mean']-history_val[f'{metric}_std'],
                marker=dict(color="#444"),
                line=dict(width=0),
                mode='lines',
                fillcolor=f'rgb{ImageColor.getcolor(color, "RGB")}'.replace('rgb','rgba').replace(')',',0.2)'),
                fill='tonexty',
                showlegend=False
            )
        )

    fig.update_layout(
        xaxis_title='Epoch',
        yaxis_title=metric.capitalize(),
        width = 960,
        height = 800
    )

    fig.show()

In [13]:
line_plot(history, 'loss')

In [14]:
line_plot(history, 'subdomain_guidance_acc')

In [15]:
line_plot(history, 'subdomain_guidance_f1')

In [16]:
line_plot(history, 'lab_guidance_acc')

In [17]:
line_plot(history, 'lab_guidance_f1')

# Calculate other metrics

## Load Predictions

In [18]:
def load_results(results_dir : str) -> dict[tuple[str, str], Tensor]:

    results = dict()

    for filename in os.listdir(results_dir):

        parts = filename.split('.')[0].split('_')
        trial = parts[-1]
        expr = '_'.join(parts[1:-1])

        results[(expr,trial)] = torch.load(os.path.join(RESULTS_DIR, filename), map_location='cpu', weights_only=False)

    return results

In [19]:
results = load_results(RESULTS_DIR)

## Label Wise & Row Wise Metrics

In [20]:
def display_mean_std_df(df : pd.DataFrame) -> pd.DataFrame:

    columns = set([
        col.replace('_mean','')
        for col in df.columns if 'mean' in col
    ])

    new_df = pd.DataFrame()

    for col in df.columns:

        if not col.endswith('_mean') and not col.endswith('_std'):
            new_df[col] = df[col]

    for col in columns:
        new_df[col] = df[f'{col}_mean'].apply(lambda x : f'{x:.2%}') + ' ± ' + df[f'{col}_std'].apply(lambda x : f'{x:.2%}')

    return new_df

In [21]:
def get_experiment_params(expr : str) -> dict:

    parts = expr.split('_')
        
    return {
        "GNN Variant" : parts[0],
        "Readout Function" : parts[1],
        "Pooling Layer" : parts[2],
        "Undersampling" : parts[3]
    }

In [22]:
def test_articles_subset(
    data : HeteroData,
    min_p : float = 0.5
) -> Tensor:
    
    fold = 0
    split = data.split
    mask = split != fold

    entity_ids, articles_ids = data['Entity', 'APPEARIN', 'Article'].edge_index
    train_entities_ids = torch.unique(entity_ids[mask[articles_ids]])

    p = scatter(
        torch.isin(entity_ids, train_entities_ids).float(),
        articles_ids,
        dim=0,
        dim_size=data['Article'].x.size(0),
        reduce='mean'
    )

    p = p[~mask]

    node_ids = torch.arange(len(p)).to(p.device)

    return node_ids[p >= min_p]

In [23]:
def get_label_wise_metrics(
    label_name : str,
    graph : HeteroData,
    results : dict[tuple[str,str],Tensor],
    labels : list[str]
) -> tuple[pd.DataFrame,pd.DataFrame]:

    label_wise_metrics = []

    data = graph['Article']

    n_labels = data[f'y_{label_name}'] if label_name != 'lab' else data[f'y_{label_name}'][:,:-1]
    n_labels = n_labels.size(-1)

    for label in range(n_labels):
            
        for (expr,trial),result in results.items():

            y = data[f'y_{label_name}'][data.split == int(trial[-1])]

            y_hat = result[label_name][data.split == int(trial[-1])].sigmoid()
            y_hard = y_hat.gt(0.5).long()

            # subset_ids = test_articles_subset(graph, min_p=0.6)

            # y_hat = y_hat[subset_ids]
            # y_hard = y_hard[subset_ids]
            # y = y[subset_ids]
            
            label_wise_metrics.append({
                "team" : label,
                "expr" : expr,
                "trial" : trial,
                "accuracy" : accuracy_score(y[:,label], y_hard[:,label]),
                "f1_score" : f1_score(y[:,label], y_hard[:,label], zero_division=0),
                "precision" : precision_score(y[:,label], y_hard[:,label], zero_division=0),
                "recall" : recall_score(y[:,label], y_hard[:,label], zero_division=0),
                "auc" : roc_auc_score(y[:,label], y_hat[:,label])
            })

    label_wise_metrics = pd.DataFrame(label_wise_metrics)
    label_wise_metrics['team'] = label_wise_metrics['team'].apply(labels.__getitem__)
    label_wise_metrics['expr'] = label_wise_metrics['expr'].apply(str)

    label_wise_metrics_agg = label_wise_metrics.drop(columns="team").groupby(["expr","trial"]).mean().reset_index().sort_values(by = 'expr')

    label_wise_metrics_mean = label_wise_metrics_agg \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .mean().add_suffix('_mean')

    label_wise_metrics_std = label_wise_metrics_agg \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .std().add_suffix('_std')

    label_wise_metrics_agg = label_wise_metrics_mean.join(label_wise_metrics_std).reset_index()

    return label_wise_metrics_agg,label_wise_metrics

In [24]:
def get_row_wise_metrics(
    label : str,
    data : HeteroData,
    results : dict[tuple[str,str],Tensor],
) -> tuple[pd.DataFrame,pd.DataFrame]:

    row_wise_metrics = []

    for (expr,trial),result in results.items():

        y = data[f'y_{label}'][data.split == int(trial[-1])]
        y_hat = result[label][data.split == int(trial[-1])].sigmoid()
        y_hard = y_hat.gt(0.5).long()

        if label == 'lab':
            y = y[:,:-1]
            y_hat = y_hat[:,:-1]
            y_hard = y_hard[:,:-1]

        row_wise_metrics.append({
            "expr" : expr,
            "trial" : trial,
            "accuracy" : accuracy_score(y, y_hard),
            "f1_score" : f1_score(y, y_hard , average='macro', zero_division=0),
            "precision" : precision_score(y, y_hard , average='macro', zero_division=0),
            "recall" : recall_score(y, y_hard , average='macro', zero_division=0),
            "auc" : roc_auc_score(y, y_hat , average='macro'),
            "hamming_loss" : hamming_loss(y, y_hard)
        })

    row_wise_metrics = pd.DataFrame(row_wise_metrics)

    row_wise_metrics_mean = row_wise_metrics \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .mean().add_suffix('_mean')

    row_wise_metrics_std = row_wise_metrics \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .std().add_suffix('_std')

    row_wise_metrics_agg = row_wise_metrics_mean.join(row_wise_metrics_std).reset_index()

    return row_wise_metrics_agg,row_wise_metrics

In [45]:
label_wise_metrics_agg,label_wise_metrics = get_label_wise_metrics('subdomain', data, results, metadata['labels']['subdomain'])
display_mean_std_df(label_wise_metrics_agg)

/home/fellah-a/anaconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning:

Only one class is present in y_true. ROC AUC score is not defined in that case.



,expr,accuracy,recall,f1_score,precision,auc
0,Conv_Mean_null_False,95.99% ± 0.09%,34.31% ± 1.91%,40.53% ± 1.17%,53.81% ± 1.24%,91.78% ± 0.50%


In [65]:
label_wise_metrics[label_wise_metrics['expr'].str.contains('Mean')].drop(columns=['trial', 'expr']).groupby(by=['team']).mean().merge(right=pd.DataFrame(data={
    "count" : data['Article'].y_subdomain.sum(dim=0)
}, index=metadata['labels']['subdomain']), left_index=True, right_index=True).corr()

,accuracy,f1_score,precision,recall,auc,count
accuracy,1.000000,-0.202218,-0.103776,-0.231129,0.517436,-0.952194
f1_score,-0.202218,1.000000,0.940527,0.992585,0.549665,0.363944
precision,-0.103776,0.940527,1.000000,0.903159,0.616596,0.237276
recall,-0.231129,0.992585,0.903159,1.000000,0.526315,0.406399
auc,0.517436,0.549665,0.616596,0.526315,1.000000,-0.367174
count,-0.952194,0.363944,0.237276,0.406399,-0.367174,1.000000


In [58]:
pd.DataFrame(data={
    "count" : data['Article'].y_subdomain.sum(dim=0)
}, index=metadata['labels']['subdomain'])

,count
real-world_data_epidemiology_&_biostatistics,1413.0
isotopic_analysis_&_metabolomics,216.0
predictive_metabolomics,193.0
multi-omics_in_kidney_diseases,96.0
cardiometabolic_multi-omics_&_pharmacology,246.0
...,...
biomechanics_&_motor_coordination_analysis,291.0
food_microbiological_safety,146.0
3d_cell_culture_&_tumor-on-chip,45.0
lipophilic_compounds_in_cosmetology,6.0


In [72]:
data['Article'].y_subdomain.shape

torch.Size([6426, 65])

In [71]:
data['Article'].y_lab.shape

torch.Size([6426, 42])

In [32]:
row_wise_metrics_agg,row_wise_metrics = get_row_wise_metrics('subdomain', data['Article'], results)
metrics = display_mean_std_df(row_wise_metrics_agg)
metrics

/home/fellah-a/anaconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning:

Only one class is present in y_true. ROC AUC score is not defined in that case.



,expr,accuracy,recall,f1_score,precision,hamming_loss,auc
0,Conv_Mean_null_False,22.95% ± 1.36%,34.31% ± 1.91%,40.53% ± 1.17%,53.81% ± 1.24%,4.01% ± 0.09%,91.97% ± 0.37%


In [33]:
label_wise_metrics_agg,label_wise_metrics = get_label_wise_metrics('lab', data, results, metadata['labels']['lab'])
display_mean_std_df(label_wise_metrics_agg)

/home/fellah-a/anaconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning:

Only one class is present in y_true. ROC AUC score is not defined in that case.

/home/fellah-a/anaconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning:

Only one class is present in y_true. ROC AUC score is not defined in that case.

/home/fellah-a/anaconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning:

Only one class is present in y_true. ROC AUC score is not defined in that case.

/home/fellah-a/anaconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning:

Only one class is present in y_true. ROC AUC score is not defined in that case.

/home/fellah-a/anaconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning:

Only one class is present 

,expr,accuracy,recall,f1_score,precision,auc
0,Conv_Mean_null_False,97.71% ± 0.02%,29.05% ± 1.31%,34.12% ± 0.91%,44.73% ± 1.05%,91.34% ± 0.82%


In [70]:
label_wise_metrics['auc'].dropna().std()

np.float64(0.061524059433649574)

In [34]:
row_wise_metrics_agg,row_wise_metrics = get_row_wise_metrics('lab', data['Article'], results)
metrics = display_mean_std_df(row_wise_metrics_agg)
metrics

/home/fellah-a/anaconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning:

Only one class is present in y_true. ROC AUC score is not defined in that case.

/home/fellah-a/anaconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning:

Only one class is present in y_true. ROC AUC score is not defined in that case.

/home/fellah-a/anaconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning:

Only one class is present in y_true. ROC AUC score is not defined in that case.

/home/fellah-a/anaconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning:

Only one class is present in y_true. ROC AUC score is not defined in that case.

/home/fellah-a/anaconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning:

Only one class is present 

,expr,accuracy,recall,f1_score,precision,hamming_loss,auc
0,Conv_Mean_null_False,39.14% ± 0.88%,29.05% ± 1.31%,34.12% ± 0.91%,44.73% ± 1.05%,2.29% ± 0.02%,nan% ± nan%


- Visualizations

In [29]:
def bar_plot(df : pd.DataFrame, metric : str) -> None:

    fig = px.bar(
        data_frame = df.sort_values(by = f'{metric}_mean'),
        x = 'expr',
        y = f'{metric}_mean',
        barmode = 'group',
        error_y = f'{metric}_std',
        text = f'{metric}_mean',
        width = 950,
        height = 800
    )

    fig.update_layout(
        xaxis_title = 'Experiment',
        yaxis_title = metric.capitalize().replace('_',' '),
    )

    fig.update_traces(
        textposition = 'outside',
        texttemplate = '%{text:.2%}'
    )

    fig.show()

In [30]:
bar_plot(label_wise_metrics_agg, 'accuracy')

In [31]:
bar_plot(label_wise_metrics_agg, 'f1_score')

In [32]:
bar_plot(row_wise_metrics_agg, 'accuracy')

# SubDomain

In [33]:
label_wise_metrics_agg,label_wise_metrics = get_label_wise_metrics('subdomain', data, results, metadata['labels']['subdomain'])
label_wise_metrics = label_wise_metrics.drop(columns=['expr', 'trial']).groupby(by=['team']).mean()

/home/fellah-a/anaconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning:

Only one class is present in y_true. ROC AUC score is not defined in that case.



In [34]:
label_wise_metrics['f1_score'].reset_index()

,team,f1_score
0,3d_cell_culture_&_tumor-on-chip,0.110232
1,ai-driven_image_&_omics_analysis,0.235636
2,ai_for_medical_imaging,0.595921
3,anti-infective_therapies,0.595461
4,antimicrobial_resistance_&_mechanistic_modeling,0.546425
...,...,...
60,therapeutic_&_diagnostic_nanomaterials,0.401891
61,transplant_immunology_&_biomarkers,0.689191
62,urban_health_&_walkability,0.758500
63,viral_genomics_&_digital_twins,0.128889


In [35]:
sizes = data['Article']['y_subdomain'].sum(dim=0).numpy()
sizes = sizes[sizes.argsort()]
scores = label_wise_metrics['f1_score'][metadata['labels']['subdomain']].to_numpy()
scores = scores[sizes.argsort()]

In [37]:
data = label_wise_metrics['f1_score'].reset_index().rename(columns={
    "team" : "SID",
    "f1_score" : "F1 Score"
})

data['text'] = data['F1 Score'].apply(lambda x : f'{(100*x):.3f}%')

fig = px.bar(
    data_frame=data,
    x="F1 Score",
    y="SID",
    text='text',
    width=950,
    height=1200,
)

fig.update_yaxes(
    tickmode="array",
    tickvals=data["SID"],
    ticktext=data["SID"],
)

fig.show()

In [41]:
(scores >= 0.5)

array([ True,  True, False, False,  True,  True,  True,  True, False,
        True, False,  True, False, False, False,  True,  True, False,
       False,  True, False, False, False, False,  True,  True,  True,
       False, False,  True,  True,  True, False, False, False, False,
        True, False, False,  True, False,  True, False,  True,  True,
       False,  True,  True, False, False, False, False, False,  True,
        True, False, False, False, False,  True,  True,  True, False,
       False,  True])

In [ ]:
fig = px.line(
    x=sizes,
    y=scores,
    width=950,
    height=600,
)

fig.show()

In [105]:
expertise = "Reconfigurable Network-on-Chip Architecture for High-Throughput Accelerator Integration"
articles = ["HT-NoC : reconfigurable High Throughput Network-on-Chip for AI dataflow accelerators", "HT-NoC: A Throughput Adaptive Multi Dataflow Network on Chip for AI Accelerators", "Accelerating the CNN Inference on FPGAs", "Multi-mode Network-on-Chip for AI dataflow accelerators", "The performance of the hypergeometric n p chart with estimated parameter"]

In [106]:
print('\\\\ \n'.join([f"{expertise} & - {a};" if i == 0 else f" & - {a};" for i,a in enumerate(articles)]) + "\\\\ \n \\hline")

Reconfigurable Network-on-Chip Architecture for High-Throughput Accelerator Integration & - HT-NoC : reconfigurable High Throughput Network-on-Chip for AI dataflow accelerators;\\ 
 & - HT-NoC: A Throughput Adaptive Multi Dataflow Network on Chip for AI Accelerators;\\ 
 & - Accelerating the CNN Inference on FPGAs;\\ 
 & - Multi-mode Network-on-Chip for AI dataflow accelerators;\\ 
 & - The performance of the hypergeometric n p chart with estimated parameter;\\ 
 \hline
